# 实验二：网页解析


## 案例一：中国大学排名爬取

目标网页：`http://www.bspider.top/gaosan/`

处理流程：获取网页 HTML → 解析排名表格 → 提取学校名称、总分、全国排名、星级排名和办学层次 → 保存到 `school.csv`。


- `get_html()`：发送 GET 请求并返回网页文本。
- `parser_university()`：使用 `table>tbody>tr` 定位表格行，再从每行的 `td` 中提取字段。第 0 列是名次，因此从第 1 列开始取 5 个字段。
- `save_csv()`：将解析结果写入 CSV 文件。

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import csv

In [2]:
def get_html(url, time=3):
    try:
        r = requests.get(url, timeout=time)
        r.encoding = r.apparent_encoding
        r.raise_for_status()
        return r.text
    except Exception as error:
        print(error)
        return None

In [3]:
def parser_university(html):
    soup = BeautifulSoup(html, "lxml")
    out_list = []

    for row in soup.select("table>tbody>tr"):
        td_html = row.select("td")
        if len(td_html) >= 6:
            row_data = [td_html[i].get_text(strip=True) for i in range(1, 6)]
            out_list.append(row_data)

    return out_list

In [4]:
def save_csv(items, path):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerows(items)

In [5]:
url = "http://www.bspider.top/gaosan/"
html = get_html(url)

if html:
    university_list = parser_university(html)
    save_csv(university_list, "school.csv")

HTTPConnectionPool(host='127.0.0.1', port=7897): Read timed out. (read timeout=3)


## 案例二：百度新闻爬取

目标网页：`http://www.bspider.top/baidunews/`

- `soup.select()`：按 CSS 选择器查找新闻链接。
- `a.get_text(strip=True)`：提取链接中的标题文本。
- `a.get("href")`：获取新闻链接；`urljoin()` 将相对链接转换为完整 URL。
- 最后将类别、标题和链接写入 CSV。

In [6]:
def parser_baidu_news(html, base_url):
    soup = BeautifulSoup(html, "lxml")
    news_list = [["新闻类别", "新闻标题", "新闻链接"]]

    selectors = [
        ("热点新闻", "div.hotnews a"),
        ("焦点新闻", "ul.focuslistnews a")
    ]

    for category, selector in selectors:
        for a in soup.select(selector):
            title = a.get_text(" ", strip=True)
            href = a.get("href")
            if title and href:
                news_list.append([category, title, urljoin(base_url, href)])

    return news_list

In [7]:
url = "http://www.bspider.top/baidunews/"
html = get_html(url)

if html:
    news_list = parser_baidu_news(html, url)
    save_csv(news_list, "baidu_news.csv")